# 🧠 Лабораторная работа 9. Локальная LLM и Qwen

Цель: отправить запрос настоящей локальной языковой модели через Ollama и изучить основные настройки inference.

> Notebook не устанавливает Ollama и не скачивает модель автоматически. Перед практикой локальный runtime и выбранная Qwen-модель должны быть установлены отдельно.


# 1. Настройки

In [ ]:
MODEL_NAME = "your-local-qwen-model"

TEMPERATURE_LOW = 0.1
TEMPERATURE_HIGH = 1.0

print("Model:", MODEL_NAME)

# 2. Проверяем Python-клиент Ollama

In [ ]:
try:
    import ollama

    print("ollama Python package доступен")
except ImportError:
    ollama = None

    print(
        "Пакет ollama не установлен. "
        "Установи его в своём окружении перед практическим запросом."
    )

# 3. Первый Prompt

In [ ]:
prompt = "Объясни простыми словами, что такое токен в языковой модели."

print(prompt)

# 4. Функция локального запроса

In [ ]:
def ask_local_model(
    prompt,
    temperature=0.2,
    system_prompt=None,
):
    if ollama is None:
        raise RuntimeError(
            "Python-пакет ollama недоступен."
        )

    messages = []

    if system_prompt:
        messages.append(
            {
                "role": "system",
                "content": system_prompt,
            }
        )

    messages.append(
        {
            "role": "user",
            "content": prompt,
        }
    )

    response = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={
            "temperature": temperature,
        },
    )

    return response["message"]["content"]

# 5. Выполняем первый запрос

In [ ]:
if ollama is not None:
    answer = ask_local_model(
        prompt,
        temperature=0.2,
    )

    print(answer)
else:
    print("Пропускаем: Ollama-клиент недоступен.")

# 6. Измеряем общее время ответа

In [ ]:
import time

if ollama is not None:
    started = time.perf_counter()

    answer = ask_local_model(
        "Что такое Transformer? Ответь в 4 предложениях.",
        temperature=0.2,
    )

    elapsed = time.perf_counter() - started

    print(answer)
    print()
    print(f"Общее время: {elapsed:.2f} сек.")
else:
    print("Пропускаем измерение.")

# 7. Сравниваем Temperature

In [ ]:
question = (
    "Придумай короткую аналогию, "
    "объясняющую механизм Attention."
)

if ollama is not None:
    print("=== LOW TEMPERATURE ===")
    print(
        ask_local_model(
            question,
            temperature=TEMPERATURE_LOW,
        )
    )

    print("\n=== HIGHER TEMPERATURE ===")
    print(
        ask_local_model(
            question,
            temperature=TEMPERATURE_HIGH,
        )
    )
else:
    print("Пропускаем запрос.")

Temperature меняет Sampling, но не обучает модель и не добавляет ей новые знания.


# 8. System Prompt

In [ ]:
system_prompt = (
    "Ты преподаватель по искусственному интеллекту. "
    "Объясняй технически корректно, но простым языком."
)

if ollama is not None:
    answer = ask_local_model(
        "Что такое embedding?",
        temperature=0.2,
        system_prompt=system_prompt,
    )

    print(answer)
else:
    print("Пропускаем запрос.")

# 9. Prompting: одна модель, разные инструкции

In [ ]:
prompts = [
    "Что такое нейросеть?",
    "Объясни нейросеть ребёнку в двух предложениях.",
    "Объясни нейросеть Python-разработчику технически.",
]

if ollama is not None:
    for current_prompt in prompts:
        print("=" * 70)
        print("PROMPT:", current_prompt)
        print()
        print(
            ask_local_model(
                current_prompt,
                temperature=0.2,
            )
        )
else:
    for current_prompt in prompts:
        print(current_prompt)

# 10. Многоходовый Chat

In [ ]:
def run_chat(messages, temperature=0.2):
    if ollama is None:
        raise RuntimeError(
            "Python-пакет ollama недоступен."
        )

    response = ollama.chat(
        model=MODEL_NAME,
        messages=messages,
        options={
            "temperature": temperature,
        },
    )

    return response["message"]["content"]


conversation = [
    {
        "role": "system",
        "content": (
            "Ты преподаватель PyTorch. "
            "Отвечай кратко."
        ),
    },
    {
        "role": "user",
        "content": "Что такое Tensor?",
    },
]

if ollama is not None:
    first_answer = run_chat(conversation)

    print("Assistant:")
    print(first_answer)

    conversation.append(
        {
            "role": "assistant",
            "content": first_answer,
        }
    )

    conversation.append(
        {
            "role": "user",
            "content": (
                "А теперь объясни, чем он "
                "отличается от списка Python."
            ),
        }
    )

    second_answer = run_chat(conversation)

    print("\nAssistant:")
    print(second_answer)
else:
    print("Пропускаем локальный Chat.")

# 11. Смотрим структуру Context

In [ ]:
for message in conversation:
    print(
        message["role"].upper(),
        "→",
        message["content"],
    )

Все эти сообщения становятся частью текущего контекста модели.

Это **не Fine-tuning**: weights модели не изменяются.


# 12. Простая оценка повторяемости

In [ ]:
test_prompt = (
    "Дай одно короткое определение "
    "градиентного спуска."
)

if ollama is not None:
    for run in range(3):
        answer = ask_local_model(
            test_prompt,
            temperature=0.1,
        )

        print(f"RUN {run + 1}:")
        print(answer)
        print()
else:
    print("Пропускаем эксперимент.")

# 13. 📌 Что нужно запомнить

```text
Prompt
↓
Tokenizer
↓
Local LLM
↓
Logits
↓
Sampling
↓
Next Token
↓
Response
```

Во время обычного inference:

```text
weights не обучаются
```


# 14. 🧩 Эксперименты

Попробуй:

- сравнить несколько Temperature;
- изменить System Prompt;
- задавать одинаковый Prompt несколько раз;
- сохранить историю Chat;
- сравнить короткий и длинный Prompt;
- измерить время для короткого и длинного ответа.


# 15. ➡️ Следующая глава

# Глава 10. AI-агенты и Tools
